In [1]:
!pip install openai pandas numpy

In [ ]:
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = 

client = OpenAI()

In [4]:
import pandas as pd

data = [
    {
        "product_id":"P1",
        "product_name":"Organic Face Cream",
        "category":"skincare",
        "price":25,
        "rating":4.5,
        "ingredients":"aloe vera vitamin c green tea",
        "reviews":"Great product very moisturizing"
    },
    {
        "product_id":"P2",
        "product_name":"Natural Shampoo",
        "category":"shampoo",
        "price":18,
        "rating":4.3,
        "ingredients":"argan oil coconut oil",
        "reviews":"Hair feels soft and healthy"
    },
    {
        "product_id":"P3",
        "product_name":"Protein Powder",
        "category":"supplements",
        "price":40,
        "rating":4.6,
        "ingredients":"whey protein cocoa",
        "reviews":"Great for workouts"
    },
    {
        "product_id":"P4",
        "product_name":"Luxury Perfume",
        "category":"fragrance",
        "price":120,
        "rating":4.8,
        "ingredients":"floral oils",
        "reviews":"Amazing fragrance"
    }
]

products = pd.DataFrame(data)

products

,product_id,product_name,category,price,rating,ingredients,reviews
0,P1,Organic Face Cream,skincare,25,4.5,aloe vera vitamin c green tea,Great product very moisturizing
1,P2,Natural Shampoo,shampoo,18,4.3,argan oil coconut oil,Hair feels soft and healthy
2,P3,Protein Powder,supplements,40,4.6,whey protein cocoa,Great for workouts
3,P4,Luxury Perfume,fragrance,120,4.8,floral oils,Amazing fragrance


In [6]:
def retrieve_products(query):

    context = products.to_dict(orient="records")

    prompt = f"""
You are an ecommerce search system.

User query:
{query}

Available products:
{context}

Return the most relevant product_ids as a Python list.

Example output:
["P1","P2"]
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content

In [7]:
def product_agent(query):

    product_context = products.to_string()

    prompt = f"""
You are an AI product assistant.

Available product data:
{product_context}

User question:
{query}

Provide helpful answer.
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content


In [8]:
product_agent("Show me affordable skincare products")

'The affordable skincare product available is:\n\n- Organic Face Cream (Price: $25, Rating: 4.5)  \n  Ingredients: aloe vera, vitamin c, green tea  \n  Review: Great product very moisturizing\n\nWould you like more details or other recommendations?'

In [8]:
golden_dataset = [
{
"query":"affordable skincare products",
"expected_category":"skincare",
"price_max":30
},
{
"query":"natural shampoo",
"expected_category":"shampoo"
},
{
"query":"protein powder for gym",
"expected_category":"supplements"
}
]

In [22]:
import ast

def evaluate_retrieval():

    correct = 0

    for test in golden_dataset:

        result = retrieve_products(test["query"])

        try:
            product_ids = ast.literal_eval(result)
        except:
            product_ids = []

        retrieved_products = products[
            products["product_id"].isin(product_ids)
        ]

        if len(retrieved_products) == 0:
            continue

        if test["expected_category"] in retrieved_products["category"].values:
            correct += 1

    accuracy = correct / len(golden_dataset)

    print("Retrieval Accuracy:", accuracy)

In [23]:
evaluate_retrieval()

Retrieval Accuracy: 1.0


In [11]:
qa_dataset = [
{
"question":"Is Organic Face Cream good?",
"expected_keywords":["positive","moisturizing","good"]
},
{
"question":"What ingredients are in Natural Shampoo?",
"expected_keywords":["argan","coconut"]
}
]

In [12]:
def evaluate_qa():

    score = 0

    for test in qa_dataset:

        answer = product_agent(test["question"])

        matches = 0

        for keyword in test["expected_keywords"]:
            if keyword in answer.lower():
                matches += 1

        if matches > 0:
            score += 1

    accuracy = score / len(qa_dataset)

    print("QA Accuracy:", accuracy)

In [13]:
evaluate_qa()

QA Accuracy: 1.0


In [14]:
def summarize_reviews(reviews):

    prompt = f"""
Summarize the sentiment of these reviews.

Reviews:
{reviews}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content

In [15]:
def test_review_sentiment():

    reviews = [
        "Great product",
        "Very good quality",
        "Loved it"
    ]

    result = summarize_reviews(reviews)

    assert "positive" in result.lower()

    print("Prompt Test Passed")

In [16]:
test_review_sentiment()

Prompt Test Passed


In [17]:
def hallucination_check(answer):

    product_text = products.to_string().lower()

    words = answer.lower().split()

    hallucinated = []

    for word in words:
        if word not in product_text:
            hallucinated.append(word)

    return hallucinated[:10]

In [18]:
ans = product_agent("What ingredients are in Organic Face Cream?")
hallucination_check(ans)

['the', 'contains', 'the', 'following', 'ingredients:', 'vera,', 'c,', 'tea.']

In [19]:
def run_full_evaluation():

    print("Running Retrieval Tests")
    evaluate_retrieval()

    print("\nRunning QA Tests")
    evaluate_qa()

    print("\nRunning Prompt Tests")
    test_review_sentiment()

In [20]:
run_full_evaluation()

Running Retrieval Tests
Retrieval Accuracy: 0.0

Running QA Tests
QA Accuracy: 1.0

Running Prompt Tests
Prompt Test Passed
